In [ ]:
# ============================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & CẤU HÌNH VRAM GPU T4
# ============================================================
import os
# 1. Cấu hình chống phân mảnh VRAM cho GPU T4 (Tránh lỗi CUDA OutOfMemory)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. Gỡ bỏ gói torchaudio thừa để triệt tiêu hoàn toàn xung đột CUDA
!pip uninstall -y torchaudio -q

# 3. Cài đặt các thư viện cần thiết cho Qwen 3-VL Vision-Language Model
!pip install -q --upgrade transformers accelerate qwen-vl-utils

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu: Thời lượng chạy (Runtime) -> Thay đổi loại thời lượng chạy -> Chọn T4 GPU.")


In [ ]:
# ============================================================
# BƯỚC 2: NẠP MÔ HÌNH QWEN 3-VL 4B INSTRUCT (CHUYÊN DỤNG VIDEO & THỊ GIÁC)
# ============================================================
import sys
import torch

# Phòng thủ kép: Tự động vô hiệu hóa torchaudio nếu phát hiện xung đột CUDA version
try:
    import torchaudio
except Exception:
    try:
        import transformers.utils.import_utils as _iu
        _iu._torchaudio_available = False
    except Exception:
        pass

try:
    from transformers import Qwen3VLForConditionalGeneration as ModelClass
except ImportError:
    try:
        from transformers import AutoModelForImageTextToText as ModelClass
    except ImportError:
        try:
            from transformers import AutoModelForVision2Seq as ModelClass
        except ImportError:
            from transformers import AutoModelForMultimodalLM as ModelClass

from transformers import AutoProcessor

# Sử dụng chính thức mô hình Qwen3-VL-4B-Instruct chuyên biệt cho Video-LLM từ Alibaba
model_id = "Qwen/Qwen3-VL-4B-Instruct"
print(f"⏳ Đang nạp mô hình {model_id} (Trọng số ~8 GB)...\n")

model = ModelClass.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# Thiết lập giới hạn pixel tối ưu cho chuỗi 8 frame trên GPU T4 (Tiết kiệm 2 GB VRAM)
min_pixels = 256 * 28 * 28
max_pixels = 512 * 28 * 28
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=min_pixels,
    max_pixels=max_pixels,
    trust_remote_code=True
)

print(f"✅ Mô hình {model_id} đã nạp thành công vào GPU T4 với kiến trúc Qwen3-VL chuyên dụng!")


In [ ]:
# ============================================================
# BƯỚC 3: XÂY DỰNG PROMPT THỜI GIAN ĐỘNG & SUY LUẬN VISUAL RELATION
# ============================================================
import os
import glob
import json
from qwen_vl_utils import process_vision_info

# 1. Thu thập và sắp xếp 8 frames ảnh theo đúng thứ tự thời gian
image_paths = sorted(
    glob.glob("/content/*.jpg") + glob.glob("/content/data/frames/video1/*.jpg"),
    key=lambda p: os.path.basename(p)
)
# Loại bỏ ảnh tạm nếu có
image_paths = [p for p in image_paths if "preview" not in p and "crop" not in p]

if len(image_paths) == 0:
    raise FileNotFoundError("❌ Không tìm thấy frames ảnh! Hãy kéo thả 8 frames từ data/frames/video1 vào thư mục /content/ trên Colab.")
print(f"✅ Đã tìm thấy {len(image_paths)} frames ảnh visual prompt.")

# 2. Tự động nạp cấu hình prompt gom nhóm (Grouped Taxonomy) từ Payload
payload_files = glob.glob("/content/*payload*.json") + glob.glob("/content/*.json")
payload_files = [p for p in payload_files if "ket_qua" not in p]

if payload_files:
    print(f"📄 Tự động tải cấu hình từ payload: {payload_files[0]}")
    with open(payload_files[0], "r", encoding="utf-8") as f:
        payload_data = json.load(f)
    system_prompt = payload_data.get("vlm_system_prompt", "")
    user_prompt = payload_data.get("vlm_user_prompt", "")
    allowed_relations = payload_data.get("allowed_relations_vocabulary_26", [])
else:
    print("ℹ️ Dùng prompt mặc định tối ưu (Chuẩn hóa gom nhóm 6 danh mục ngữ nghĩa - Cách 3):")
    allowed_objects_60 = ['bird', 'cattle', 'dog', 'horse', 'lizard', 'rabbit', 'sheep', 'snake', 'turtle', 'chicken', 'duck', 'cat', 'pig', 'goat', 'person', 'child', 'bicycle', 'bus', 'car', 'motorcycle', 'train', 'baby_seat', 'baby_walker', 'stop_sign', 'traffic_light', 'truck', 'scooter', 'ball', 'skateboard', 'sofa', 'bread', 'cake', 'dish', 'fruits', 'vegetables', 'backpack', 'camera', 'cellphone', 'handbag', 'laptop', 'suitcase', 'bat', 'racket', 'toy', 'bottle', 'chair', 'cup', 'electric_fan', 'faucet', 'sink', 'oven', 'microwave', 'refrigerator', 'screen', 'stool', 'table', 'toilet', 'guitar', 'piano', 'bench']
    allowed_relations = ['bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit', 'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)', 'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave']
    
    system_prompt = (
        "You are an advanced Video Visual Relation Detection (VidVRD) AI for surveillance analytics. You are given a temporal sequence of video frames with numbered visual marks [ID] identifying subjects and objects. Your task is to detect all active visual relations occurring between the marked entities over time.\n"
        "\n"
        "STRICT CONSTRAINTS:\n"
        "1. You MUST strictly select relation predicates ONLY from these 26 predefined categories: ['bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit', 'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)', 'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave'].\n"
        "2. Entity subject and object classes belong strictly to the 60 predefined categories: ['bird', 'cattle', 'dog', 'horse', 'lizard', 'rabbit', 'sheep', 'snake', 'turtle', 'chicken', 'duck', 'cat', 'pig', 'goat', 'person', 'child', 'bicycle', 'bus', 'car', 'motorcycle', 'train', 'baby_seat', 'baby_walker', 'stop_sign', 'traffic_light', 'truck', 'scooter', 'ball', 'skateboard', 'sofa', 'bread', 'cake', 'dish', 'fruits', 'vegetables', 'backpack', 'camera', 'cellphone', 'handbag', 'laptop', 'suitcase', 'bat', 'racket', 'toy', 'bottle', 'chair', 'cup', 'electric_fan', 'faucet', 'sink', 'oven', 'microwave', 'refrigerator', 'screen', 'stool', 'table', 'toilet', 'guitar', 'piano', 'bench'].\n"
        "\n"
        "3. GROUPED SEMANTIC TAXONOMY (CLOSED VOCABULARY):\n"
        "[A. Physical Contact & Manipulation]\n"
        "  • touch: Direct physical contact between entities (e.g., hand touching another person's arm or shoulder) without grasping.\n"
        "  • hold: Physically grasping and supporting an entity with hands or arms (hands must actively grip the entity; standing near an object resting on the ground is NOT hold).\n"
        "  • carry: Supporting and transporting an entity while in motion.\n"
        "  • grab: Sudden, rapid taking hold of an entity.\n"
        "  • lift: Raising an entity upward from a lower position.\n"
        "[B. Spatial Separation & Security]\n"
        "  • get_off: Subject who was previously in physical possession, holding, or carrying an entity actively releases, departs from, or leaves it stationary (a passerby with no prior possession or physical contact CANNOT get_off an entity).\n"
        "  • lean_on: Resting body weight against an object or person for support.\n"
        "[C. Physical Force & Motion Transfer]\n"
        "  • push: Applying force to move an entity away from subject.\n"
        "  • pull: Applying force to move an entity closer to subject.\n"
        "  • throw: Propelling an object through the air by hand.\n"
        "  • hit: Striking an entity with sudden physical force.\n"
        "  • kick: Forcefully striking an entity with a foot.\n"
        "  • knock: Striking a surface to produce sound or attract attention.\n"
        "[D. Vehicle & Animal Mobility (VEHICLE/ANIMAL ONLY)]\n"
        "  • get_on: Mounting or entering a vehicle or rideable animal.\n"
        "  • drive: Operating and controlling a motorized vehicle.\n"
        "  • ride: Sitting on and traveling via a vehicle or animal.\n"
        "[E. Interpersonal & Social Interaction]\n"
        "  • shake_hand_with: Mutual hand grasping between persons in greeting.\n"
        "  • hug: Wrapping arms tightly around another person.\n"
        "  • kiss: Touching another person with lips.\n"
        "  • wave: Moving hand back and forth as a visual signal/greeting.\n"
        "[F. Specialized Actions]\n"
        "  • bite: Using teeth to grip or cut an object.\n"
        "  • lick: Passing tongue over an entity.\n"
        "  • feed: Giving food to another person or animal.\n"
        "  • clean: Removing dirt, stains, or marks from an object.\n"
        "  • cut: Dividing or penetrating an object using a sharp tool.\n"
        "  • play(instrument): Actively playing a musical instrument.\n"
        "\n"
        "4. SYSTEMATIC INTERACTION RULES:\n"
        "   - Person-Person interactions: Identify active physical contact or intentional social interaction. NEVER use 'get_off' for Person-Person pairs.\n"
        "   - Person-Object interactions: Only predict manipulation verbs ('hold', 'carry') if a person is physically grasping the object with hands or arms. Merely standing or walking near an entity resting stationary on the ground is NOT holding.\n"
        "   - SPECIAL RULE FOR 'get_off': In this taxonomy, use 'get_off' ONLY when a person who previously held, carried, or was in physical possession of an entity actively releases, departs from, or leaves it stationary. A passerby who never had possession or physical contact with the entity CANNOT have a 'get_off' relation.\n"
        "   - Vehicle Rules: Predicates like 'get_on', 'ride', 'drive' MUST ONLY be used if the object is explicitly a vehicle (bicycle, car, motorcycle, bus, train) or an animal (horse).\n"
        "   - STRICT NEGATIVE PAIR FILTERING: If an entity pair has no active physical contact or confirmed relationship (e.g., a person merely walking past an object on the floor without touching or prior possession), you MUST completely OMIT that pair. Do NOT output any triplet entry for that pair in the 'triplets' array.\n"
        "   - Ground Truth Fidelity: Strictly report visual facts. Do not hallucinate actions that are not visible. If an object remains visible on the floor in the final frames, it is NOT picked up.\n"
        "5. Output format MUST be strictly a valid JSON object matching this schema:\n"
        "{\n"
        "  \"triplets\": [\n"
        "    {\n"
        "      \"subject\": \"[ID]\",\n"
        "      \"relation\": \"<predicate>\",\n"
        "      \"object\": \"[ID]\",\n"
        "      \"reason\": \"<brief explanation of why this relation is selected based on visual evidence>\"\n"
        "    }\n"
        "  ]\n"
        "}\n"
        "6. DO NOT output any markdown code blocks, explanations, or conversational text. Output ONLY the raw JSON object.\n"
    )
    
    user_prompt = (
        "Analyze all provided sequential frames of this surveillance video clip.\n"
        "Examine active interactions between the marked entities [ID] across time.\n\n"
        "\n"
        "Perform a systematic check across the full time duration:\n"
        "- Examine ALL Person-Person combinations.\n"
        "- Examine ALL Person-Object combinations.\n"
        "\n"
        "\n"
        "PREDEFINED RELATION TAXONOMY (CLOSED VOCABULARY):\n"
        "Every predicate in the 'relation' field MUST be an exact string match selected strictly from the 26 allowed categories defined in the taxonomy. All out-of-vocabulary verbs are strictly prohibited.\n"
        "- To denote a person releasing, departing from, or leaving an entity stationary that they previously possessed, use 'get_off'.\n"
        "- STRICT FILTERING: If two entities have no physical contact or active interaction (such as a passerby walking past an object without touching or prior possession), omit that pair entirely. Do NOT output any triplet for that pair.\n"
        "\n"
    )

# 3. Chuẩn bị nội dung tuần tự kèm mốc thời gian rõ ràng (Interleaved Temporal Anchoring)
user_content = []
for idx, p in enumerate(image_paths, 1):
    base_fn = os.path.basename(p)
    ts = base_fn.split("_")[-1].replace(".jpg", "") if "_" in base_fn and "s.jpg" in base_fn else f"{idx}s"
    user_content.append({"type": "text", "text": f"[Frame {idx} at timestamp {ts}]:"})
    user_content.append({"type": "image", "image": p})
user_content.append({"type": "text", "text": "\n" + user_prompt})

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# 4. Xử lý dữ liệu đầu vào và suy luận trên GPU T4 (Siêu tốc 20s)
text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

proc_kwargs = {"text": [text_prompt], "padding": True, "return_tensors": "pt"}
if image_inputs is not None and len(image_inputs) > 0:
    proc_kwargs["images"] = image_inputs
if video_inputs is not None and len(video_inputs) > 0:
    proc_kwargs["videos"] = video_inputs

import gc
gc.collect()
torch.cuda.empty_cache()

inputs = processor(**proc_kwargs).to("cuda")

with torch.no_grad():
    # Pure Greedy Search: Bảo toàn token khóa JSON chính xác
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=False
    )

generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
raw_output = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

# 5. Trích xuất và kiểm tra mảng JSON kết quả kèm cột Reason
clean_text = raw_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
try:
    parsed_json = json.loads(clean_text)

    if isinstance(parsed_json, dict):
        if summary:
            print(f"🎬 Quan sát thời gian (Temporal Summary): {summary}\n")
        triplets = parsed_json.get("triplets", [])
    elif isinstance(parsed_json, list):
        triplets = parsed_json
    else:
        triplets = []

    # Lưu kết quả độc lập ra file JSON
    with open("/content/ket_qua_vlm.json", "w", encoding="utf-8") as f:
        json.dump(triplets, f, indent=2, ensure_ascii=False)

    print("=" * 80)
    print("KẾT QUẢ SUY LUẬN VLM (PREDICTED RELATION TRIPLETS + REASONING):")
    print("=" * 80)
    print(json.dumps(triplets, indent=2, ensure_ascii=False))
    print("-" * 80)
    print(f"{'Subject':<10} | {'Relation':<16} | {'Object':<10} | {'Status':<10} | Reason / Explanation")
    print("-" * 80)
    for t in triplets:
        sub = t.get("subject", "")
        rel = t.get("relation", "")
        obj = t.get("object", "")
        reason = t.get("reason", "N/A")
        is_valid = rel in allowed_relations if allowed_relations else True
        status = "[OK]" if is_valid else "[X] Invalid"
        print(f"{sub:<10} | {rel:<16} | {obj:<10} | {status:<10} | {reason}")
    print("=" * 80)
    print(f"✅ Đã lưu kết quả tự động vào: /content/ket_qua_vlm.json ({len(triplets)} triplets)")
except Exception as e:
    print("Raw output từ mô hình:", raw_output)
    print(f"Lỗi phân tích JSON: {e}")
